# U2-01 · Explorar, visualizar y preparar los datos de las bicicletas

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 2*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Fanaee-T, H. y Gama, J. (2014). Event labeling combining ensemble detectors and background knowledge. *Progress in Artificial Intelligence*, 2, 113–127. Bike Sharing, UCI, CC BY 4.0, https://doi.org/10.24432/C5W894

## 2.2 · Examinar los datos

`g1_explorar.py`

In [ ]:
import pandas as pd

h = pd.read_csv("hour.csv", parse_dates=["dteday"])
print(h.shape, "| de", h.dteday.min().date(), "a",
      h.dteday.max().date())
print("valores ausentes:", int(h.isna().sum().sum()),
      "| filas repetidas:", int(h.duplicated().sum()))
print("casual + registered = cnt:",
      bool((h.casual + h.registered == h.cnt).all()))
print(h[["temp", "hum", "windspeed", "cnt"]].describe()
      .loc[["mean", "min", "max"]].round(3).to_string())
horas = h.groupby("dteday").size()
print(f"días: {len(horas)}, horas esperadas {len(horas) * 24},"
      f" registradas {len(h)}")
print("días con menos de 20 horas:")
print(horas[horas < 20].rename(lambda d: d.date()).to_string())
print("situación meteorológica:",
      h.weathersit.value_counts().sort_index().to_dict())
print("humedad 0:", (h.hum == 0).sum(), "horas, todas el",
      h[h.hum == 0].dteday.dt.strftime("%Y-%m-%d").unique()[0])

## 2.3 · Visualizar los datos de entrada

`g2_graficos.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

h = pd.read_csv("hour.csv", parse_dates=["dteday"])
fig, ej = plt.subplots(2, 2, figsize=(4.4, 3.4), layout="constrained")
perfil = h.pivot_table(index="hr", columns="workingday",
                       values="cnt", aggfunc="mean")
ej[0, 0].plot(perfil[1], label="laborable")
ej[0, 0].plot(perfil[0], label="festivo o fin de semana")
ej[0, 0].set_title("Alquileres por hora"); ej[0, 0].legend(fontsize=5)
dia = h.groupby("dteday").cnt.sum()
ej[0, 1].plot(dia.index, dia.rolling(7).mean(), lw=0.8)
ej[0, 1].set_title("Alquileres al día (media de 7 días)")
ej[0, 1].xaxis.set_major_locator(mdates.MonthLocator([1, 7]))
ej[0, 1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
ej[1, 0].scatter(h.temp * 41, h.cnt, s=0.3, alpha=0.2)
ej[1, 0].set_title("Alquileres frente a temperatura (°C)")
cols = ["temp", "atemp", "hum", "windspeed", "casual",
        "registered", "cnt"]
c = h[cols].corr()
im = ej[1, 1].imshow(c, cmap="RdBu_r", vmin=-1, vmax=1)
ej[1, 1].set_xticks(range(7), cols, rotation=90, fontsize=5)
ej[1, 1].set_yticks(range(7), cols, fontsize=5)
ej[1, 1].set_title("Correlaciones")
fig.colorbar(im, ax=ej[1, 1], shrink=0.8); plt.show()
print("hora punta laborable:", perfil[1].idxmax(), "h,",
      round(perfil[1].max()), "| festivo:", perfil[0].idxmax(), "h,",
      round(perfil[0].max()))
print("correlación con cnt:", c.cnt.drop("cnt").round(2).to_dict())
print(f"día de más alquileres: {dia.idxmax().date()} ({dia.max()})")

## 2.4 · Preparar y reducir las variables

`g3_preparar.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_regression

h = pd.read_csv("hour.csv", parse_dates=["dteday"])
d = pd.DataFrame({                       # deshacer la normalización
    "temp_c": h.temp * 41, "sensacion_c": h.atemp * 50,
    "humedad": h.hum * 100, "viento_kmh": h.windspeed * 67})
d.loc[h.dteday == "2011-03-10", "humedad"] = np.nan   # sensor roto
d["humedad"] = d.humedad.interpolate()    # rellenar con vecinos
d["tiempo"] = h.weathersit.clip(upper=3)  # 4 casi no existe: a 3
d["hora_sen"] = np.sin(2 * np.pi * h.hr / 24)  # hora cíclica:
d["hora_cos"] = np.cos(2 * np.pi * h.hr / 24)  # 23 h junto a 0 h
d = d.join(pd.get_dummies(h.weekday, prefix="dia", dtype=int))
print(d.shape, "| humedad media el 10/3/2011:",
      round(d.humedad[h.dteday == "2011-03-10"].mean(), 1))
print("correlación temperatura-sensación:",
      round(d.temp_c.corr(d.sensacion_c), 3))
X = d.drop(columns="sensacion_c").assign(hr=h.hr)
mi = mutual_info_regression(X, h.cnt, random_state=0)
print("información mutua con cnt (más alta, más útil):")
print(pd.Series(mi, X.columns).sort_values(ascending=False)
      .head(6).round(3).to_string())
ent = h.dteday < "2012-07-01"             # separar por fecha
print(f"entrenamiento {ent.sum()} horas, prueba {(~ent).sum()}")

**Prueba.** Calcula cuántas horas faltan en cada mes. ¿Qué meses concentran las horas perdidas?